In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Users/gainrakesh2001@gmail.com/consolidated_pipeline/1_setup/utilities

In [0]:
print(bronze_schema,silver_schema,gold_schema)

In [0]:
dbutils.widgets.text("catalog","fmcg","Catalog")
dbutils.widgets.text("data_source","orders","Data Source")

In [0]:
 # created widgets becuase dynamically change the catalog,data_source
# suppose this can run different environment like dev ,test,prod e.g (dev_fmcg,test_fmcg,prod_fmcg) and (prod_fmcg,test_fmcg,dev_fmcg) etc
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f"/Volumes/workspace/default/sportsbar_raw/{data_source}/"
lading_path = f"{base_path}/landing"
processed_path = f"{base_path}/processed"
print(catalog,data_source)
print(base_path)

In [0]:
bronze_table = f"{catalog}.{bronze_schema}.{data_source}"
silver_table = f"{catalog}.{silver_schema}.{data_source}"
gold_table = f"{catalog}.{gold_schema}.sb_dm_{data_source}"
print(bronze_table,silver_table,gold_table)

In [0]:
# loading the data into in df
df = spark.read.format("csv")\
        .option("header",True)\
        .option("inferSchema",True)\
        .load(f"{base_path}/landing")

df = df.withColumn("read_timestamp",F.current_timestamp())
df = df.select("*","_metadata.file_name","_metadata.file_size")


In [0]:
display(df.limit(10))

In [0]:
df.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed","true")\
    .mode("append")\
    .saveAsTable(bronze_table)

In [0]:
# Let move all the file from landing to processed and i want to empty landing folder becuase i want to read only new file from landing folder , this only batch data processing
files = dbutils.fs.ls(lading_path)
for file_info in files:
  dbutils.fs.mv(file_info.path,
                f"{processed_path}/{file_info.name}" , True)


### Silver Processing

In [0]:
df_orders = spark.sql(f"select * from {bronze_table}")
df_orders.show(5)

In [0]:
# keep only rows where order_qty is present
df_orders= df_orders.filter(F.col("order_qty").isNotNull())

# clean customer_id - > keep numeric , else set to 999999
df_orders = df_orders.withColumn(
    "customer_id" , F.when(F.col("customer_id").rlike("^[0-9]+$"), F.col("customer_id")).otherwise("999999").cast("string")
)

# remove the weekday name from the date text
# "Tuesday, July 01, 2025" - > "July 01,2025"

df_orders = df_orders.withColumn(
    "order_placement_date", F.regexp_replace(F.col("order_placement_date"), "^[A-Za-z]+,\s*", "")
)

date_format = ["yyyy/MM/dd","dd/MM/yyyy","yyyy-MM-dd","dd-MM-yyyy"]

df_orders = df_orders.withColumn(
    "order_placement_date" ,
    F.coalesce(
        F.try_to_date(F.col("order_placement_date"), "yyyy/MM/dd"),
        F.try_to_date(F.col("order_placement_date"), "dd/MM/yyyy"),
        F.try_to_date(F.col("order_placement_date"), "MMMM dd, yyyy"),
        F.try_to_date(F.col("order_placement_date"), "dd-MM-yyyy")
    )
    
)




In [0]:
df_orders = df_orders.dropDuplicates(["order_id","order_placement_date","customer_id","product_id","order_qty"])

df_orders = df_orders.withColumn("product_id",F.col("product_id").cast('string'))


In [0]:
df_orders.agg(
    F.min("order_placement_date").alias("min_date"),
    F.max("order_placement_date").alias("max_date")
).show()

In [0]:
display(df_orders.limit(5))

In [0]:
# Now i will add the product_code in this table because product_code cloumn is now PK in product table , now will join the product table and this df_orders based on product_id column
df_products = spark.table("fmcg.silver.products")
display(df_products.limit(5))

In [0]:
df_joined = df_orders.join(df_products,df_orders.product_id == df_products.product_id,"inner").select(df_orders["*"], df_products["product_code"])



In [0]:
display(df_joined.limit(5))

In [0]:
df_joined.write\
        .format("delta")\
        .option("delta.enableChangeDataFeed", "true")\
        .option("mergeSchema","true")\
        .mode("overwrite")\
        .saveAsTable(silver_table)

In [0]:
# Upsert the data into the delta table 
delta_table = DeltaTable.forName(spark, silver_table)

delta_table.alias("silver").merge(
    df_joined.alias("bronze"),
    "silver.order_id = bronze.order_id and silver.order_placement_date = bronze.order_placement_date and silver.customer_id = bronze.customer_id and silver.product_id = bronze.product_id"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

### Gold processing 

In [0]:
df_gold = spark.sql(f""" select order_id , order_placement_date as date , customer_id as customer_code , product_code , product_id , order_qty as sold_quantity from {silver_table}  
                    """)
df_gold.show(3)


In [0]:
df_gold.write\
        .format("delta")\
        .option("delta.enableChangeDataFeed", "true")\
        .option("mergeSchema","true")\
        .mode("overwrite")\
        .saveAsTable(gold_table)

In [0]:
%sql
-- Enable column mapping to support DROP COLUMN
ALTER TABLE fmcg.gold.sb_dm_orders
SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name');

-- Now drop the sold_qty column
ALTER TABLE fmcg.gold.sb_dm_orders
DROP COLUMN sold_qty;

In [0]:
# Upsert the data into the delta table 
delta_table = DeltaTable.forName(spark, gold_table)

delta_table.alias("gold").merge(
    df_gold.alias("source"),
    "gold.order_id = source.order_id and source.date = gold.date and source.product_code = gold.product_code and source.customer_code = gold.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

### Merge with Parent Company

In [0]:
df_child = spark.sql(f""" select date , product_code , customer_code , sold_quantity from {gold_table}
                     """)
df_child.show(10)

In [0]:
# Upsert the data into the delta table 
gold_parent_delta = DeltaTable.forName(spark, f"{catalog}.{gold_schema}.fact_orders") # target table

gold_parent_delta.alias("parent_gold").merge(
    df_child.alias("chile_gold"),
    "parent_gold.date = chile_gold.date and parent_gold.product_code = chile_gold.product_code and parent_gold.customer_code = chile_gold.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()